## Setup

In [10]:
from googleapiclient.discovery import build
import json
import re
from collections import Counter
from datetime import datetime
from googleapiclient.discovery import build
import pandas as pd
import numpy as np
import glob

In [11]:
with open ("../performative.txt", "r") as f:
    lines = f.readlines()
    API_KEY = lines[2].strip()
    
yt = build("youtube", "v3", developerKey=API_KEY)

### Static/Maps

In [12]:
GENRE_MAP = {
    "1":  "Film & Animation",
    "2":  "Autos & Vehicles",
    "10": "Music",
    "15": "Pets & Animals",
    "17": "Sports",
    "19": "Travel & Events",
    "20": "Gaming",
    "22": "People & Blogs",
    "23": "Comedy",
    "24": "Entertainment",
    "25": "News & Politics",
    "26": "Howto & Style",
    "27": "Education",
    "28": "Science & Technology",
    "29": "Nonprofits & Activism",
}

In [13]:
QUERIES = [
    "video essay film analysis",
    "philosophy explained documentary",
    "internet culture deep dive",
    "history documentary essay channel",
    "game design analysis",
]
MAX_PER_QUERY = 50  # how many channels to collect per keyword/phrase


## Scripts

In [ ]:
def search_channels_queries(query, max_results, seen):
    collected = []
    next_page = None

    while len(collected) < max_results:
        resp = yt.search().list(
            part="snippet",
            q = query, 
            type="channel",
            relevanceLanguage = "en",
            maxResults=50,
            pageToken = next_page,
        ).execute()

        for item in resp["items"]:
            cid = item["snippet"]["channelId"]
            if cid not in seen:
                seen.add(cid)
                collected.append({
                    "channel_id":   item["snippet"]["channelId"],
                    "channel_name": item["snippet"]["title"],
                })
            if len(collected) >= max_results:
                break
        
        next_page = resp.get("nextPageToken")
        if not next_page:
            break
    return collected

In [ ]:
def search_channels_playlist(playlist_id, max_results, seen):
    collected = []
    next_page = None

    while len(collected) < max_results:
        resp = yt.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page,
        ).execute()

        for item in resp["items"]:
            cid = item["snippet"]["channelId"]
            if cid not in seen:
                seen.add(cid)
                collected.append({
                    "channel_id":   item["snippet"]["channelId"],
                    "channel_name": item["snippet"]["title"],
                })
            if len(collected) >= max_results:
                break
        
        next_page = resp.get("nextPageToken")
        if not next_page:
            break
    return collected

## Running

In [ ]:
all_channels = []
seen = set()

for query in QUERIES:
    print(f"Searching: {query}")
    results = search_channels_queries(query, MAX_PER_QUERY, seen)
    print(f" got {len(results)} channels")
    all_channels.extend(results)

with open("../data/processed/ve_channels/ve_channels_queries.json", "r") as f:
    existing = json.load(f)
combined = existing + all_channels
with open("../data/processed/ve_channels_queries.json", "w") as f:
    json.dump(all_channels, f, indent=2)

Searching: video essay film analysis
 got 50 channels
Searching: philosophy explained documentary
 got 50 channels
Searching: internet culture deep dive
 got 50 channels
Searching: history documentary essay channel
 got 50 channels
Searching: game design analysis
 got 25 channels


In [ ]:
all_channels = []
seen = set()

results = search_channels_queries("PLXz6Pf8rae1mIGQ8IdpggWP2RmW1wmXsa", MAX_PER_QUERY, seen)

with open("../data/processed/ve_channels/ve_channels_playlist.json", "r") as f:
    existing = json.load(f)
combined = existing + all_channels
with open("../data/processed/ve_channels_playlist.json", "w") as f:
    json.dump(all_channels, f, indent=2)

## Viewing Data

In [ ]:
with open("../data/processed/ve_channels_queries.json", "r") as f:
    data = json.load(f)

In [24]:
df = pd.DataFrame(data)
df.head(50)

,channel_id,channel_name
0,UCqgh7QNlVV8kv3LBTJZYWrg,The Film Essay
1,UCjFqcJQXGZ6T6sxyFB-5i6A,Every Frame a Painting
2,UCr8r7UVsDaRiIQSVUHVBd_A,Nikki Carreon
3,UCjKSoJlPgcK6BmoSqXpj5xQ,Action Button
4,UCUyvQV2JsICeLZP4c_h40kA,Thomas Flight
5,UCQUicR6JrQjRXse6x4M6lvw,Full Fat Videos
6,UC02m7n7CJNYez3Ht0eejvZw,You Have Been Watching Films
7,UC0SrgAKjftBuzHMfRZSjt7g,Lancelloti
8,UCM0V8r4kuIWIl6Sy-NHj2lg,Quinton Reviews
9,UCF-5VNvG0-pm44qSgVrO6Yg,About Movies
